To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth your local device, follow [our guide](https://docs.unsloth.ai/get-started/install-and-update). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & [how to save it](#Save)

### News


New 3x faster training & 30% less VRAM. New kernels, padding-free & packing. [Blog](https://docs.unsloth.ai/new/3x-faster-training-packing)

You can now train with 500K context windows on a single 80GB GPU. [Blog](https://docs.unsloth.ai/new/500k-context-length-fine-tuning)

Unsloth's [Docker image](https://hub.docker.com/r/unsloth/unsloth) is here! Start training with no setup & environment issues. [Read our Guide](https://docs.unsloth.ai/new/how-to-train-llms-with-unsloth-and-docker).

New in Reinforcement Learning: [FP8 RL](https://docs.unsloth.ai/new/fp8-reinforcement-learning) • [Vision RL](https://docs.unsloth.ai/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://docs.unsloth.ai/basics/memory-efficient-rl) (faster, less VRAM RL) • [gpt-oss RL](https://docs.unsloth.ai/new/gpt-oss-reinforcement-learning)

Visit our docs for all our [model uploads](https://docs.unsloth.ai/get-started/all-our-models) and [notebooks](https://docs.unsloth.ai/get-started/unsloth-notebooks).


### Installation

In [2]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.33.post1")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2 && pip install --no-deps trl==0.22.2

### Unsloth

In [3]:
from unsloth import FastSentenceTransformer

fourbit_models = [
    "unsloth/all-MiniLM-L6-v2",
    "unsloth/embeddinggemma-300m",
    "unsloth/Qwen3-Embedding-4B",
    "unsloth/Qwen3-Embedding-0.6B",
    "unsloth/all-mpnet-base-v2",
    "unsloth/gte-modernbert-base",
    "unsloth/bge-m3"

] # More models at https://huggingface.co/unsloth

model = FastSentenceTransformer.from_pretrained(
    model_name = "unsloth/Qwen3-Embedding-0.6B",
    max_seq_length = 512,   # Choose any for long context!
    load_in_4bit = False,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

==((====))==  Unsloth 2026.1.4: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

We now add LoRA adapters so we only need to update a small amount of parameters!

In [4]:
model = FastSentenceTransformer.get_peft_model(
    model,
    r = 32, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
    task_type = "FEATURE_EXTRACTION"
)

Unsloth: Making `model.base_model.model` require gradients


<a name="Data"></a>
### Data Prep
We now use the `electroglyph/technical` dataset.

In [5]:
from datasets import load_dataset
dataset = load_dataset("FreedomIntelligence/huatuo_encyclopedia_qa", split="train")

README.md: 0.00B [00:00, ?B/s]

train_datasets.jsonl:   0%|          | 0.00/604M [00:00<?, ?B/s]

validation_datasets.jsonl: 0.00B [00:00, ?B/s]

test_datasets.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/362420 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [6]:
def format_huatuo(example):
    return {
        "anchor": example['questions'][0][0],
        "positive": example['answers'][0]
    }

dataset = dataset.map(
    format_huatuo,
    remove_columns=dataset.column_names
)

Map:   0%|          | 0/362420 [00:00<?, ? examples/s]

Let's take a look at the dataset structure:

In [7]:
print("Dataset examples:")
for i in range(6):
    print(dataset[i])

Dataset examples:
{'anchor': '曲匹地尔片的用法用量', 'positive': '注意：同种药品可由于不同的包装规格有不同的用法或用量。本文只供参考。如果不确定，请参看药品随带的说明书或向医生询问。口服。一次50～100mg（1-2片），3次/日，或遵医嘱。'}
{'anchor': '三期梅毒多久能治愈吗', 'positive': '梅毒是一种进展十分缓慢的疾病，包括早期梅毒，晚期梅毒等等，一般来说，梅毒病程超过两年的就是晚期梅毒了，而晚期梅毒的治疗难度是比较大的，三期梅毒能完全治愈吗?该如何进行治疗?一、对于三期梅毒的治疗，中西各有妙招，希望患者在医生的指导下选择适合自己的治疗方法。被确诊为三期梅毒的患者很关心这个问题，梅毒需要讲究科学的治疗方法，三期梅毒患者如果有一个良好的心态，积极配合医生治疗，还有很有可能治愈的，具体治疗方法我们看看下文的介绍。二、三期梅毒又称晚期梅毒，是因梅毒初期没有进行治疗或者治疗不彻底所造成。好发于40-50岁之间。该期梅毒不仅局限于皮肤和粘膜，也可侵犯任何内脏器官和组织。三期梅毒可发生在感染后2年以上，一般多发于感染后3～4年。病程漫长，可持续10～30年。未经治愈的二期梅毒中约有1/3的病人可发展为晚期活动性梅毒;另有一部分患者不出现晚期梅毒症状，只是梅毒血清反应持续阳性，为晚期潜伏梅毒;也有一部分患者可以自愈。三、晚期梅毒可以通过药物治疗控制病情，但是治愈的可能性是比较小的，治疗还是有必要的，通过有效的治疗可以缓解患者的症状，杀灭一部分的梅毒螺旋体。虽然治愈的可能性不高，但是患者还是应当坚持治疗。通过上文的简单介绍，相信大家对于三期梅毒多久能治愈这一方面的问题已经有了自己的答案。对于梅毒患者来说，无论患病多少年，都应在确诊后及时进行正规的治疗处理。正规的治疗可以缓解患者的病情，患者还是应当坚持进行，并积极配合医生的治疗，说不定治愈的可能就发生了。'}
{'anchor': '肝癌术后饮食', 'positive': '肝脏恶性肿瘤可分为原发性和继发性。原发性肝脏恶性肿瘤源于肝脏的上皮或间质组织。前者被称为原发性肝癌，是我国发病率高、危害大的恶性肿瘤。后者被称为肉瘤，与原发性肝癌相比相对较少。肝癌术后饮食注意什么呢？肝脏手术给人体带来巨大的创伤。剩余肝细胞的再生、增值和伤口愈合需要消耗大量的能量和蛋

## Baseline Inference
Let's test the base model before fine-tuning to see how it performs on our specific domain.

In [19]:
from sentence_transformers import util
import torch

def test_inference(model, run_name="Run"):
    """Test model with a query and candidate sentences"""

    query = "阿莫西林胶囊主要用于治疗哪些疾病？"
    candidates = [
        # 1. [正确答案]：核心是抗感染
        "阿莫西林适用于敏感菌(不产β内酰胺酶菌株)所致的呼吸道感染、泌尿生殖道感染、皮肤软组织感染等。",

        # 2. [硬负例 - 用法用量]：关键词完全匹配，但语义意图完全不同
        "口服。成人一次0.5g，每6～8小时1次，一日剂量不超过4g。",

        # 3. [硬负例 - 不良反应]：也是关于阿莫西林的，但不是治什么
        "恶心、呕吐、腹泻及假膜性肠炎等胃肠道反应。皮疹、药物热和哮喘等过敏反应。",

        # 4. [中等负例 - 相似领域]：也是治感冒发烧的，但原理不同（比如这是退烧药）
        "用于缓解轻至中度疼痛如头痛、关节痛、偏头痛、牙痛、肌肉痛、神经痛、痛经。",

        # 5. [简单负例 - 完全无关]：心血管疾病，领域跨度大
        "用于治疗高血压、心绞痛。口服，起始剂量10mg，每日一次。"
    ]
    """
    query="阿莫西林治什么？"
    candidates=["阿莫西林适用于呼吸道感染。","阿莫西林口服，一次0.5g。"]
    """
    with torch.inference_mode():
      with torch.autocast(device_type="cuda", dtype=torch.float32):
          query_emb = model.encode(query, convert_to_tensor=True)
          candidate_embs = model.encode(candidates, convert_to_tensor=True)
    scores = util.cos_sim(query_emb, candidate_embs)[0]

    results = []
    for i, score in enumerate(scores):
        results.append((candidates[i], score.item()))
    results.sort(key=lambda x: x[1], reverse=True)

    print(f"\n--- {run_name} Results for query: '{query}' ---")
    for text, score in results:
        print(f"{score:.4f} | {text}")

test_inference(model, run_name="Pre-Training")


--- Pre-Training Results for query: '阿莫西林胶囊主要用于治疗哪些疾病？' ---
0.8155 | 阿莫西林适用于敏感菌(不产β内酰胺酶菌株)所致的呼吸道感染、泌尿生殖道感染、皮肤软组织感染等。
0.4845 | 用于缓解轻至中度疼痛如头痛、关节痛、偏头痛、牙痛、肌肉痛、神经痛、痛经。
0.3753 | 用于治疗高血压、心绞痛。口服，起始剂量10mg，每日一次。
0.2545 | 口服。成人一次0.5g，每6～8小时1次，一日剂量不超过4g。
0.1304 | 恶心、呕吐、腹泻及假膜性肠炎等胃肠道反应。皮疹、药物热和哮喘等过敏反应。


<a name="Train"></a>
### Train the model
Now let's train our model. We use `MultipleNegativesRankingLoss`

 This loss function uses other positives in the same batch as negative examples, which is efficient for contrastive learning.

 We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

In [9]:
from sentence_transformers import (
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    losses
)
from sentence_transformers.training_args import BatchSamplers
from unsloth import is_bf16_supported

# This will use other positives in the same batch as negative examples
loss = losses.MultipleNegativesRankingLoss(model)

trainer = SentenceTransformerTrainer(
    model = model,
    train_dataset = dataset,
    loss = loss,
    args=SentenceTransformerTrainingArguments(
        # num_train_epochs=1,
        max_steps = 60,
        per_device_train_batch_size = 64,
        gradient_accumulation_steps = 2, # Use GA to mimic batch size!
        learning_rate = 3e-5,
        fp16=not is_bf16_supported(),
        bf16 = is_bf16_supported(),
        logging_steps=1,
        warmup_ratio = 0.03,
        report_to="none", # Use TrackIO/WandB etc
        output_dir = "output",
        lr_scheduler_type = "constant_with_warmup",
        # Because we have duplicate anchors in the dataset, we don't want
        # to accidentally use them for negative examples
        batch_sampler = BatchSamplers.NO_DUPLICATES,
    ),

)

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

In [10]:
# @title Show current memory stats
import torch
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.741 GB.
1.236 GB of memory reserved.


Let's train the model! To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [11]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 362,420 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 64 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (64 x 2 x 1) = 128
 "-____-"     Trainable parameters = 20,185,088 of 615,961,600 (3.28% trained)


Step,Training Loss
1,0.245800
2,0.375400
3,0.225600
4,0.202500
5,0.164100
6,0.308700
7,0.189000
8,0.187700
9,0.125300
10,0.240000


In [12]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

1042.3466 seconds used for training.
17.37 minutes used for training.
Peak reserved memory = 5.705 GB.
Peak reserved memory for training = 4.469 GB.
Peak reserved memory % of max memory = 38.702 %.
Peak reserved memory for training % of max memory = 30.317 %.


<a name="Inference"></a>
### Inference
Let's run the model after training to see the improvements!

In [13]:
test_inference(model, run_name="Post-Training")


--- Post-Training Results for query: '阿莫西林治什么？' ---
0.7735 | 阿莫西林适用于呼吸道感染。
0.7088 | 阿莫西林口服，一次0.5g。


<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit, scroll down!

In [16]:
model.save_pretrained("lora_model")  # Local saving
# model.push_to_hub("your_name/lora_model", token = "...") # Online saving

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if False:
    from unsloth import FastSentenceTransformer
    model = FastSentenceTransformer.from_pretrained(
        "model",
        # for_inference=True
    )

### Saving to float16 for vLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens. See [our docs](https://docs.unsloth.ai/basics/inference-and-deployment) for more deployment options.

In [ ]:
# Merge to 16bit
if False:
    model.save_pretrained_merged("model", tokenizer=model.tokenizer, save_method = "merged_16bit",)
if False: # Pushing to HF Hub
    model.push_to_hub_merged("hf/model", tokenizer=model.tokenizer, save_method = "merged_16bit", token = "")

# Just LoRA adapters
if False:
    model.save_pretrained("model")
if False: # Pushing to HF Hub
    model.push_to_hub("hf/model", token = "")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [Wiki page](https://docs.unsloth.ai/basics/inference-and-deployment/saving-to-gguf#locally)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.


In [ ]:
# Save to 8bit Q8_0
if False:
    model.save_pretrained_gguf("model",)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False:
    model.push_to_hub_gguf("hf/model", token = "")

# Save to 16bit GGUF
if True:
    model.save_pretrained_gguf("model", quantization_method = "f16")
if False: # Pushing to HF Hub
    model.push_to_hub_gguf("lastmass/Qwen3-Embedding-Medical-0.6B", quantization_method = "f16", token = "")

# Save to q4_k_m GGUF
if False:
    model.save_pretrained_gguf("model", quantization_method = "q4_k_m")
if False: # Pushing to HF Hub
    model.push_to_hub_gguf("hf/model", quantization_method = "q4_k_m", token = "")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "hf/model", # Change hf to your username!
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "", # Get a token at https://huggingface.co/settings/tokens
    )

Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `/content/model`: 100%|██████████| 1/1 [00:13<00:00, 13.64s/it]


Successfully copied all 1 files from cache to `/content/model`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:08<00:00,  8.53s/it]


Unsloth: Merge process complete. Saved to `/content/model`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['f16'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: All required system packages already installed!
Unsloth: Install llama.cpp and building - please wait 1 to 3 minutes
Unsloth: Cloning llama.cpp repository
Unsloth: Install GGUF and other packages
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['Qwen3-Embedding-0.6B.F16.gguf']
Unsloth: Model f

In [16]:
import os
import shutil
from google.colab import drive

# 1. 挂载 Google Drive (如果已经挂载过，这行会自动跳过)
drive.mount('/content/drive')

# ================= 配置区域 =================
# 源文件夹：Unsloth 默认保存模型的路径
source_dir = "/content/model"

# 目标文件夹：在 Drive 上创建一个专门的文件夹来存放这次训练的所有文件
# 建议给文件夹起个带日期的名字，防止以后混淆
dest_dir = "/content/drive/MyDrive/Unsloth_Models/Qwen3_Full_Export"
# ===========================================

def copy_folder_to_drive(src, dst):
    try:
        if not os.path.exists(src):
            print(f"❌ 错误：源文件夹不存在 -> {src}")
            return

        print(f"🚀 开始将整个文件夹复制到 Google Drive...")
        print(f"📂 源路径: {src}")
        print(f"📂 目标路径: {dst}")

        # dirs_exist_ok=True 允许目标文件夹已存在（会合并/覆盖），防止报错
        shutil.copytree(src, dst, dirs_exist_ok=True)

        print("-" * 50)
        print(f"✅ 成功！文件夹已完整备份。")
        print(f"查看路径: {dst}")

        # 顺便列出一下复制过去的文件数量，让你安心
        num_files = sum([len(files) for r, d, files in os.walk(dst)])
        print(f"📊 共包含 {num_files} 个文件")

    except Exception as e:
        print(f"❌ 复制过程中发生错误: {str(e)}")

# 执行复制
copy_folder_to_drive(source_dir, dest_dir)

Mounted at /content/drive
🚀 开始将整个文件夹复制到 Google Drive...
📂 源路径: /content/model
📂 目标路径: /content/drive/MyDrive/Unsloth_Models/Qwen3_Full_Export
--------------------------------------------------
✅ 成功！文件夹已完整备份。
查看路径: /content/drive/MyDrive/Unsloth_Models/Qwen3_Full_Export
📊 共包含 17 个文件


In [17]:
from huggingface_hub import HfApi, create_repo

# ================= 配置区域 =================
# 1. 请在这里填入你的 Hugging Face Token (必须是 Write 权限)
HF_TOKEN = ""

# 2. 仓库 ID 和本地路径
REPO_ID = "lastmass/Qwen3-Embedding-Medical-0.6B"
LOCAL_FOLDER = "/content/model"
# ===========================================

def push_to_hub():
    if not HF_TOKEN:
        print("❌ 错误：请先在 HF_TOKEN 变量中填入你的 Token！")
        return

    try:
        api = HfApi(token=HF_TOKEN)

        print(f"🚀 正在连接 Hugging Face Hub...")

        # 1. 确保仓库存在 (如果不存在会自动创建，exist_ok=True 表示如果已存在也不报错)
        # 默认创建为公开仓库，如果需要私有，请加参数 private=True
        repo_url = api.create_repo(
            repo_id=REPO_ID,
            exist_ok=True,
            repo_type="model"
        )
        print(f"✅ 仓库已就绪: {repo_url}")

        # 2. 上传整个文件夹
        print(f"📤 开始推送文件夹: {LOCAL_FOLDER} -> {REPO_ID}")
        print("   (文件较大时可能需要几分钟，请耐心等待进度条...)")

        api.upload_folder(
            folder_path=LOCAL_FOLDER,
            repo_id=REPO_ID,
            repo_type="model",
            commit_message="Upload Unsloth fine-tuned embedding model (GGUF + F16)"
        )

        print("-" * 50)
        print(f"🎉 成功！模型已发布。")
        print(f"🔗 访问地址: https://huggingface.co/{REPO_ID}")

    except Exception as e:
        print(f"❌ 推送过程中发生错误: {str(e)}")

# 执行推送
push_to_hub()

🚀 正在连接 Hugging Face Hub...
✅ 仓库已就绪: https://huggingface.co/lastmass/Qwen3-Embedding-Medical-0.6B
📤 开始推送文件夹: /content/model -> lastmass/Qwen3-Embedding-Medical-0.6B
   (文件较大时可能需要几分钟，请耐心等待进度条...)


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 25.4kB / 80.8MB            

  ...t/model/model.safetensors:   3%|2         | 33.4MB / 1.19GB            

  ...3-Embedding-0.6B.F16.gguf:   2%|2         | 25.2MB / 1.20GB            

  ...tent/model/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

--------------------------------------------------
🎉 成功！模型已发布。
🔗 访问地址: https://huggingface.co/lastmass/Qwen3-Embedding-Medical-0.6B


In [18]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
